# Practice: classical screening filters, pads, recall and pruning

A sandbox for `orbit_correction.screening`. It builds a small **synthetic** LEO catalog,
screens it all-pairs to get a reference answer, then runs the three classical filters
(apogee/perigee, orbit path, time) on *noisy* elements and shows how the pad trades
recall against the fraction of pairs pruned.

Nothing here touches real data, `src/` or `data/`. Change any parameter and re-run.

**What the noise stands for.** The reference trajectories come from exact elements
(two-body plus J2 secular drift). The filters see the same elements with random errors
added, a crude stand-in for the gap between an element set and the truth propagator. With
exact elements the filters lose nothing at zero pad (see
`tests/test_screening.py::test_filters_lose_nothing_when_elements_are_exact`); the pad
exists to absorb what the filter's inputs get wrong. The error sizes below are
illustrative, not measured.

In [1]:
import numpy as np

from orbit_correction import screening as S

DAY = S.DAY_S
T0 = 7.88e8                      # TT seconds since J2000.0 (any epoch works here)
N_OBJECTS = 400
WINDOW_DAYS = 2.0
THRESHOLD_KM = 10.0              # truth threshold for this toy catalog (more events than 5 km)

t = T0 + np.arange(0.0, WINDOW_DAYS * DAY + 30.0, 60.0)
el_true = S.synthetic_leo_elements(N_OBJECTS, T0, seed=4)
truth_trajs = S.trajectories_from_elements(el_true, t, j2=True)
truth = S.screen(truth_trajs, THRESHOLD_KM)
print(f"{N_OBJECTS} objects, {S.pair_count(N_OBJECTS):,} pairs, {truth.events.num_rows} "
      f"reference events in {WINDOW_DAYS:g} days ({truth.stats['wall_s']:.1f} s)")
truth.events.slice(0, 5).to_pylist()

400 objects, 79,800 pairs, 62 reference events in 2 days (1.6 s)


[{'norad_id_1': 281,
  'norad_id_2': 352,
  'tca_tt_s': 788003268.2016655,
  'tau_s': 3268.201665520668,
  'miss_km': 0.5260690190566817,
  'rel_speed_km_s': 3.5946160843806516,
  'threshold_km': 10.0},
 {'norad_id_1': 144,
  'norad_id_2': 331,
  'tca_tt_s': 788004842.2386194,
  'tau_s': 4842.2386194467545,
  'miss_km': 9.015997475102028,
  'rel_speed_km_s': 13.373351645379492,
  'threshold_km': 10.0},
 {'norad_id_1': 49,
  'norad_id_2': 293,
  'tca_tt_s': 788005124.2697613,
  'tau_s': 5124.269761323929,
  'miss_km': 3.5496480099397347,
  'rel_speed_km_s': 14.439219133687514,
  'threshold_km': 10.0},
 {'norad_id_1': 366,
  'norad_id_2': 386,
  'tca_tt_s': 788005717.9164315,
  'tau_s': 5717.916431546211,
  'miss_km': 9.55030445400317,
  'rel_speed_km_s': 10.228094566491759,
  'threshold_km': 10.0},
 {'norad_id_1': 116,
  'norad_id_2': 175,
  'tca_tt_s': 788007104.870323,
  'tau_s': 7104.870322942734,
  'miss_km': 3.9059401062700903,
  'rel_speed_km_s': 11.700757442449147,
  'threshold_k

## The filters' inputs: elements with errors

`SIGMA_A_KM` shifts each orbit's size (radial error, and through the mean motion a growing
along-track drift). `SIGMA_ANGLE_DEG` perturbs inclination, node and argument of
perigee. `SIGMA_PHASE_DEG` perturbs the mean anomaly at epoch (along-track error at the
start).

In [2]:
SIGMA_A_KM = 1.0
SIGMA_ANGLE_DEG = 0.01
SIGMA_PHASE_DEG = 0.05

def noisy(el, seed=0):
    rng = np.random.default_rng(seed)
    n = len(el)
    return S.KeplerElements(
        el.norad_id, el.epoch_tt_s,
        el.a_km + rng.normal(0, SIGMA_A_KM, n),
        np.clip(el.e + rng.normal(0, 1e-4, n), 0.0, 0.5),
        el.i_rad + np.radians(rng.normal(0, SIGMA_ANGLE_DEG, n)),
        el.raan_rad + np.radians(rng.normal(0, SIGMA_ANGLE_DEG, n)),
        el.argp_rad + np.radians(rng.normal(0, SIGMA_ANGLE_DEG, n)),
        el.mean_anomaly_rad + np.radians(rng.normal(0, SIGMA_PHASE_DEG, n)),
    )

el_seen = noisy(el_true)
# Along-track drift implied by the semimajor-axis error after the window, for scale:
n0 = np.sqrt(S.MU_EARTH_KM3_S2 / el_true.a_km**3)
drift_km = 1.5 * n0 * SIGMA_A_KM / el_true.a_km * WINDOW_DAYS * DAY * el_true.a_km
print(f"typical along-track drift from a {SIGMA_A_KM} km error over the window: "
      f"{np.median(drift_km):.0f} km")

typical along-track drift from a 1.0 km error over the window: 269 km


## Pad sweep

The filters run in the usual order (cheapest first): apogee/perigee, then orbit path on
what survives, then the time filter. The same distance pad goes to all three here, and the
time filter also gets a time pad. Recall is per event: a reference event is kept if its
pair survives.

In [3]:
def run_chain(el, pad_km, time_pad_s):
    ids, q, Q = S.apsides_from_elements(el)
    ap = S.apogee_perigee_filter(ids, q, Q, threshold_km=THRESHOLD_KM, pad_km=pad_km)
    op = S.orbit_path_filter(el, [t[0], t[t.size // 2], t[-1]], threshold_km=THRESHOLD_KM,
                             pad_km=pad_km, prior=ap.mask)
    tf = S.time_filter(el, (t[0], t[-1]), threshold_km=THRESHOLD_KM, pad_km=pad_km,
                       time_pad_s=time_pad_s, prior=op.mask)
    return ap, op, tf

print(f"{'pad km':>7} {'time pad s':>10} | {'A/P recall':>10} {'pruned':>7} | "
      f"{'+OP recall':>10} {'pruned':>7} | {'+T recall':>9} {'pruned':>7}")
for pad_km, time_pad_s in [(0, 0), (2, 5), (5, 10), (10, 30), (20, 60), (40, 120)]:
    row = []
    for res in run_chain(el_seen, pad_km, time_pad_s):
        m = S.filter_metrics(truth.events, res)
        row += [m["event_recall"], res.pruned_fraction]
    print(f"{pad_km:>7} {time_pad_s:>10} | {row[0]:>10.3f} {row[1]:>7.3f} | "
          f"{row[2]:>10.3f} {row[3]:>7.3f} | {row[4]:>9.3f} {row[5]:>7.3f}")

 pad km time pad s | A/P recall  pruned | +OP recall  pruned | +T recall  pruned


      0          0 |      1.000   0.894 |      0.952   0.955 |     0.581   0.997


      2          5 |      1.000   0.889 |      1.000   0.949 |     0.968   0.993


      5         10 |      1.000   0.882 |      1.000   0.939 |     1.000   0.991


     10         30 |      1.000   0.871 |      1.000   0.923 |     1.000   0.986


     20         60 |      1.000   0.848 |      1.000   0.895 |     1.000   0.976


     40        120 |      1.000   0.805 |      1.000   0.845 |     1.000   0.952


Things to look for:

* Recall rises with the pad and the pruned fraction falls: the operating characteristic
  Alfano and Finkleman (2013) plot.
* The apogee/perigee filter is cheap and needs only a pad as large as the radial error.
* The time filter prunes the most, but its misses come from timing, which the distance pad
  covers poorly (a 1 km semimajor-axis error becomes tens of km along-track in two days).
  Try raising only `time_pad_s`.

## Which events does a filter lose?

In [4]:
ap, op, tf = run_chain(el_seen, pad_km=5.0, time_pad_s=10.0)
for res in (ap, op, tf):
    m = S.filter_metrics(truth.events, res)
    print(f"{res.name:>15}: lost {len(m['lost_pairs'])} of {m['n_pairs']} reference pairs; "
          f"recall by horizon "
          + ", ".join(f"<= {h / DAY:g} d: {v['event_recall']:.2f}"
                      for h, v in m['recall_by_horizon'].items() if v['n_events']))

 apogee_perigee: lost 0 of 57 reference pairs; recall by horizon <= 1 d: 1.00, <= 3 d: 1.00, <= 7 d: 1.00
     orbit_path: lost 0 of 57 reference pairs; recall by horizon <= 1 d: 1.00, <= 3 d: 1.00, <= 7 d: 1.00
           time: lost 0 of 57 reference pairs; recall by horizon <= 1 d: 1.00, <= 3 d: 1.00, <= 7 d: 1.00


## Swap in your own filter

Edit this copy of the apogee/perigee rule (for example, drop the threshold from the
comparison, or pad perigee and apogee differently) and compare it with the module's
version. The module is not modified.

In [5]:
def my_apogee_perigee(ids, q, Q, threshold_km, pad_km):
    i, j = np.triu_indices(len(ids), 1)
    gap = np.maximum(q[i], q[j]) - np.minimum(Q[i], Q[j])
    return S.PairMask(ids, gap <= pad_km)          # threshold deliberately left out

ids, q, Q = S.apsides_from_elements(el_seen)
for pad in (0.0, 5.0, 10.0, 20.0):
    mine = my_apogee_perigee(ids, q, Q, THRESHOLD_KM, pad)
    ref = S.apogee_perigee_filter(ids, q, Q, threshold_km=THRESHOLD_KM, pad_km=pad).mask
    print(f"pad {pad:>4}: module recall {S.filter_metrics(truth.events, ref)['event_recall']:.3f}"
          f" (pruned {ref.pruned_fraction:.3f}); mine "
          f"{S.filter_metrics(truth.events, mine)['event_recall']:.3f} "
          f"(pruned {mine.pruned_fraction:.3f})")

pad  0.0: module recall 1.000 (pruned 0.894); mine 0.871 (pruned 0.916)
pad  5.0: module recall 1.000 (pruned 0.882); mine 0.935 (pruned 0.905)
pad 10.0: module recall 1.000 (pruned 0.871); mine 1.000 (pruned 0.894)
pad 20.0: module recall 1.000 (pruned 0.848); mine 1.000 (pruned 0.871)


## A padded, horizon-dependent screen

A cheap screen uses `threshold + pad(tau)`. Screen the *noisy* orbits with a pad that grows
with time from the epoch and score the events against the reference. Recall per event
needs a TCA tolerance: here 120 s.

With these element errors the along-track drift reaches hundreds of km in two days (printed
above), so recall only recovers with very large pads, and false alarms explode on the way.
That is the problem a learned correction attacks: less error, smaller pad, fewer false
alarms at the same recall. Try `SIGMA_A_KM = 0.1` and re-run the notebook.

In [6]:
seen_trajs = S.trajectories_from_elements(el_seen, t, j2=True)

for pad_per_day in (0.0, 20.0, 60.0, 150.0, 300.0):
    def threshold(tau, p=pad_per_day):
        return THRESHOLD_KM + 1.0 + p * np.maximum(tau, 0) / DAY

    cheap = S.screen(seen_trajs, threshold)
    sc = S.screening_metrics(truth.events, cheap.events, tca_tol_s=120.0,
                             wall_s=cheap.stats["wall_s"])
    print(f"pad 1 km + {pad_per_day:>4} km/day: event recall {sc.event_recall:.3f}, pair recall "
          f"{sc.pair_recall:.3f}, false-alarm events {sc.false_alarm_events}, {sc.wall_s:.1f} s")

pad 1 km +  0.0 km/day: event recall 0.226, pair recall 0.193, false-alarm events 54, 1.2 s


pad 1 km + 20.0 km/day: event recall 0.468, pair recall 0.421, false-alarm events 764, 1.2 s


pad 1 km + 60.0 km/day: event recall 0.677, pair recall 0.649, false-alarm events 4561, 1.2 s


pad 1 km + 150.0 km/day: event recall 0.887, pair recall 0.877, false-alarm events 23220, 1.4 s


pad 1 km + 300.0 km/day: event recall 1.000, pair recall 1.000, false-alarm events 74187, 1.8 s
